# 🚀 GenAI Assignment 6: Enhanced Flux Studio

This notebook provides a professional interface for the **FLUX.1-schnell** model.
**Features:**
- 🎨 **Style Presets** (Cinematic, Anime, Cyberpunk, etc.)
- 📐 **Aspect Ratio Control** (Square, Portrait, Landscape)
- 🎲 **Random Prompt Generator**
- ⚙️ **Advanced Settings** (Seed, Steps, Negative Prompt)

In [1]:
# 1. Install required libraries
!pip install -q huggingface_hub gradio pillow

import gradio as gr
from huggingface_hub import InferenceClient
from PIL import Image
from google.colab import userdata
import random
import io

In [2]:
# 2. Setup Hugging Face Client
# Model: FLUX.1-schnell (optimized for speed and quality)
try:
    HF_TOKEN = userdata.get('HUGGINGFACEHUB_API_TOKEN')
    client = InferenceClient("black-forest-labs/FLUX.1-schnell", token=HF_TOKEN)
    print("✅ Client set up successfully!")
except Exception as e:
    print("❌ Error: Please set your HF_TOKEN in Colab Secrets (Key: HUGGINGFACEHUB_API_TOKEN).")

✅ Client set up successfully!


In [3]:
# 3. Configuration: Styles & Aspect Ratios

# Aspect Ratios (Width, Height)
ASPECT_RATIOS = {
    "Square (1:1)": (1024, 1024),
    "Cinematic (16:9)": (1280, 720),
    "Portrait (9:16)": (720, 1280),
    "Landscape (4:3)": (1152, 864),
    "Instagram (4:5)": (864, 1080)
}

# Style Prompt Templates
STYLES = {
    "None (Raw)": "{prompt}",
    "Cinematic": "cinematic film still of {prompt}, highly detailed, high budget hollywood movie, bokeh, atmospheric, sharp focus",
    "Anime": "anime style artwork of {prompt}, studio ghibli, vibrant colors, detailed background, cell shading",
    "Cyberpunk": "cyberpunk style {prompt}, neon lights, futuristic, high tech, dark atmosphere, rain, reflection",
    "3D Render": "3d render of {prompt}, unreal engine 5, octane render, ray tracing, hyperrealistic, 8k",
    "Oil Painting": "oil painting of {prompt}, heavy brushstrokes, texture, classic art style, masterpiece",
    "Minimalist": "minimalist design of {prompt}, flat colors, simple shapes, clean lines, vector art style"
}

# Random Prompts for Inspiration
RANDOM_PROMPTS = [
    "A futuristic city floating in the clouds at sunset",
    "A cute robot gardener watering plants in a greenhouse",
    "An astronaut playing chess with an alien on Mars",
    "A magical library where books fly like birds",
    "A steampunk coffee machine making espresso"
]

In [4]:
# 4. Define Generation Logic
def generate_enhanced_image(prompt, negative_prompt, style_name, ratio_name, seed, num_steps):
    try:
        # 1. Apply Style Template
        style_template = STYLES.get(style_name, "{prompt}")
        final_prompt = style_template.format(prompt=prompt)

        # 2. Append Negative Prompt if provided (Flux handles this via prompt engineering mostly)
        if negative_prompt:
            final_prompt += f" [AVOID: {negative_prompt}]"

        # 3. Get Dimensions
        width, height = ASPECT_RATIOS.get(ratio_name, (1024, 1024))

        # 4. Handle Random Seed
        if seed == -1:
            seed = random.randint(0, 1000000)

        print(f"🎨 Generating: '{final_prompt[:50]}...' | Size: {width}x{height} | Seed: {seed}")

        # 5. Generate Image
        image = client.text_to_image(
            final_prompt,
            height=height,
            width=width,
            num_inference_steps=num_steps,
            seed=seed
        )
        return image, f"✅ Done! Used Seed: {seed}"
    except Exception as e:
        return None, f"❌ Error: {str(e)}"

def get_random_prompt():
    return random.choice(RANDOM_PROMPTS)

In [ ]:
theme = gr.themes.Ocean()

with gr.Blocks(title="Flux GenAI Studio", theme=theme) as demo:
    gr.Markdown(
        """
        # 🎨 Flux GenAI Studio
        ### High-Speed Image Generation with **FLUX.1-schnell**
        Create stunning visuals with customizable styles, ratios, and advanced settings.
        """
    )

    with gr.Row():
        # LEFT COLUMN: Inputs
        with gr.Column(scale=4):
            with gr.Group():
                prompt_input = gr.Textbox(
                    label="Your Imagination",
                    placeholder="Describe what you want to see...",
                    lines=3
                )

                with gr.Row():
                    style_dropdown = gr.Dropdown(
                        label="Art Style",
                        choices=list(STYLES.keys()),
                        value="None (Raw)"
                    )
                    ratio_dropdown = gr.Dropdown(
                        label="Aspect Ratio",
                        choices=list(ASPECT_RATIOS.keys()),
                        value="Square (1:1)"
                    )

            # Advanced Settings Accordion
            with gr.Accordion("⚙️ Advanced Settings", open=False):
                neg_prompt = gr.Textbox(
                    label="Negative Prompt (Things to avoid)",
                    placeholder="blur, low quality, distorted, ugly",
                    value=""
                )
                with gr.Row():
                    seed_input = gr.Number(label="Seed (-1 for Random)", value=-1, precision=0)
                    steps_slider = gr.Slider(minimum=1, maximum=10, value=4, step=1, label="Inference Steps")

            with gr.Row():
                clear_btn = gr.ClearButton([prompt_input, neg_prompt])
                random_btn = gr.Button("🎲 Surprise Me", variant="secondary")
                generate_btn = gr.Button("🚀 Generate Image", variant="primary", scale=2)

        # RIGHT COLUMN: Output
        with gr.Column(scale=5):
            image_output = gr.Image(label="Generated Masterpiece", type="pil", interactive=False)
            status_text = gr.Textbox(label="Status Info", show_label=False, interactive=False)

    # Event Wiring
    random_btn.click(get_random_prompt, outputs=prompt_input)

    generate_btn.click(
        fn=generate_enhanced_image,
        inputs=[prompt_input, neg_prompt, style_dropdown, ratio_dropdown, seed_input, steps_slider],
        outputs=[image_output, status_text]
    )

# Launch
demo.launch(share=True, debug=True)

/tmp/ipython-input-2865379526.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Flux GenAI Studio", theme=theme) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e0f87fed3193fa69f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎨 Generating: 'cute cat...' | Size: 1024x1024 | Seed: 143986


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🎨 Generating: 'anime style artwork of cute cat, studio ghibli, vi...' | Size: 1024x1024 | Seed: 512702
🎨 Generating: 'anime style artwork of cute cat, studio ghibli, vi...' | Size: 1280x720 | Seed: 8223
🎨 Generating: 'oil painting of cute cat, heavy brushstrokes, text...' | Size: 1280x720 | Seed: 885567
🎨 Generating: '3d render of cute cat, unreal engine 5, octane ren...' | Size: 1280x720 | Seed: 40515
🎨 Generating: '3d render of A steampunk coffee machine making esp...' | Size: 1280x720 | Seed: 512674
